### **Minimum temperatures**

Run from this folder with the data-engineering kernel. Input: `1800.csv`.

Source: https://github.com/PacktPublishing/Frank-Kanes-Taming-Big-Data-with-Apache-Spark-and-Python/blob/master/1800.csv

In [1]:
import findspark
findspark.init()

In [2]:
from pyspark import SparkConf, SparkContext

conf = SparkConf().setMaster("local").setAppName("MinTemperatures")
sc = SparkContext.getOrCreate(conf)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/05 10:19:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/05 10:19:44 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
def parseLine(line):
    fields = line.split(',')
    stationID = fields[0]
    entryType = fields[2]
    temperature = float(fields[3]) * 0.1 * (9.0 / 5.0) + 32.0
    return (stationID, entryType, temperature)

lines = sc.textFile("./data/1800.csv")
parsedLines = lines.map(parseLine)
minTemps = parsedLines.filter(lambda x: "TMIN" in x[1])
stationTemps = minTemps.map(lambda x: (x[0], x[2]))
minTemps = stationTemps.reduceByKey(lambda x, y: min(x, y))
results = minTemps.collect()

for result in results:
    print(result[0] + "\t{:.2f}F".format(result[1]))

ITE00100554	5.36F
EZE00100082	7.70F


In [4]:
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder\
    .appName("sampleSpark")\
    .master("local[*]")\
    .getOrCreate()

In [6]:
data = [("Alice", 25), ("Bob", 30), ("Charlie", 45)]
cols = ["Name", "Age"]

df = spark.createDataFrame(data, schema=cols)
df.show()

+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    Bob| 30|
|Charlie| 45|
+-------+---+



In [10]:
df.filter(df["Age"] > 28).show()

+-------+---+
|   Name|Age|
+-------+---+
|    Bob| 30|
|Charlie| 45|
+-------+---+



**User Defined Functions**

In [12]:
from pyspark.sql.types import IntegerType

def square(x):
    return x*x

spark.udf.register("square", square, IntegerType())

<function __main__.square(x)>